In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- classification_iloc ---
FIX_CLASSIFICATION_ILOC_DS = SimpleNamespace(X=pd.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.]}),y=pd.DataFrame({"y":[0,1,0,1,0]}),w=pd.DataFrame({"w":[1.,1.,1.,1.,1.]}),x_columns=["x1","x2"],y_columns=["y"],w_columns=["w"],to_pandas=lambda:pd.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.],"y":[0,1,0,1,0],"w":[1.,1.,1.,1.,1.]}),to_polars=lambda:pl.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.],"y":[0,1,0,1,0],"w":[1.,1.,1.,1.,1.]}),filter=lambda mask: SimpleNamespace(X=pd.DataFrame({"x1":[1.,2.],"x2":[4.,5.]}),y=pd.DataFrame({"y":[0,1]}),w=pd.DataFrame({"w":[1.,1.]}),x_columns=["x1","x2"],y_columns=["y"],w_columns=["w"],to_pandas=lambda:pd.DataFrame({"x1":[1.,2.],"x2":[4.,5.],"y":[0,1],"w":[1.,1.]}),to_polars=lambda:pl.DataFrame({"x1":[1.,2.],"x2":[4.,5.],"y":[0,1],"w":[1.,1.]}),save=lambda p:None),save=lambda p:None)
FIX_CLASSIFICATION_ILOC_TRAIN_IDX = [0, 1, 2]
FIX_CLASSIFICATION_ILOC_VALID_IDX = [0, 1, 2]

# --- classification_pred_concat ---
FIX_CLASSIFICATION_PRED_CONCAT_ACCURACY_SCORE = lambda *a, **k: 0.75
FIX_CLASSIFICATION_PRED_CONCAT_DS = SimpleNamespace(X=pd.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.]}),y=pd.DataFrame({"y":[0,1,0,1,0]}),w=pd.DataFrame({"w":[1.,1.,1.,1.,1.]}),x_columns=["x1","x2"],y_columns=["y"],w_columns=["w"],to_pandas=lambda:pd.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.],"y":[0,1,0,1,0],"w":[1.,1.,1.,1.,1.]}),to_polars=lambda:pl.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.],"y":[0,1,0,1,0],"w":[1.,1.,1.,1.,1.]}),filter=lambda mask: SimpleNamespace(X=pd.DataFrame({"x1":[1.,2.],"x2":[4.,5.]}),y=pd.DataFrame({"y":[0,1]}),w=pd.DataFrame({"w":[1.,1.]}),x_columns=["x1","x2"],y_columns=["y"],w_columns=["w"],to_pandas=lambda:pd.DataFrame({"x1":[1.,2.],"x2":[4.,5.],"y":[0,1],"w":[1.,1.]}),to_polars=lambda:pl.DataFrame({"x1":[1.,2.],"x2":[4.,5.],"y":[0,1],"w":[1.,1.]}),save=lambda p:None),save=lambda p:None)
FIX_CLASSIFICATION_PRED_CONCAT_F1_SCORE = lambda *a, **k: 0.75
FIX_CLASSIFICATION_PRED_CONCAT_PRECISION_SCORE = lambda *a, **k: 0.75
FIX_CLASSIFICATION_PRED_CONCAT_PRED = np.array([0.3, 0.7, 0.4, 0.8, 0.5])
FIX_CLASSIFICATION_PRED_CONCAT_RECALL_SCORE = lambda *a, **k: 0.75
FIX_CLASSIFICATION_PRED_CONCAT_ROC_AUC_SCORE = lambda *a, **k: 0.75
FIX_CLASSIFICATION_PRED_CONCAT_VALID_IDX = [0, 1, 2, 3, 4]  # matches len(pred)

def make_classification_pred_concat_base_dfs_pd():
    return [pd.DataFrame({"y":[0,1,0],"w":[1.,1.,1.],"pred":[0.3,0.7,0.4]}),
            pd.DataFrame({"y":[1,0],"w":[1.,1.],"pred":[0.8,0.5]})]

def make_classification_pred_concat_base_dfs_pl():
    return [pl.DataFrame({"y":[0,1,0],"w":[1.,1.,1.],"pred":[0.3,0.7,0.4]}),
            pl.DataFrame({"y":[1,0],"w":[1.,1.],"pred":[0.8,0.5]})]

print("✅ Fixtures loaded")

In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_classification_iloc(ds, train_idx, valid_idx):
    train_X = ds.X.iloc[train_idx]
    train_y = ds.y.iloc[train_idx].to_numpy().reshape(-1)
    valid_X = ds.X.iloc[valid_idx]
    valid_w = ds.w.iloc[valid_idx].to_numpy().reshape(-1)
    valid_y = ds.y.iloc[valid_idx].to_numpy().reshape(-1)
    return valid_y

def before_classification_pred_concat(accuracy_score, base_dfs, ds, f1_score, precision_score, pred, recall_score, roc_auc_score, valid_idx):
    pred_df = pd.DataFrame(
        {"index": ds.y.index[valid_idx], "pred": pred}
    ).set_index("index")
    base_df = pd.merge(
        ds.y.rename(columns={ds.y_columns[0]: "y"}),
        ds.w.rename(columns={ds.w_columns[0]: "w"}),
        left_index=True,
        right_index=True,
    )
    base_dfs.append(
        pd.merge(
            pred_df,
            base_df,
            left_index=True,
            right_index=True,
        )
    )
    base_df = pd.concat(base_dfs)
    _metrics = {
    "roc_auc": roc_auc_score(base_df.y, base_df.pred),
    "accuracy": accuracy_score(base_df.y, base_df.pred > 0.5),
    "precision": precision_score(base_df.y, base_df.pred > 0.5),
    "recall": recall_score(base_df.y, base_df.pred > 0.5),
    "f1": f1_score(base_df.y, base_df.pred > 0.5),
    }
    return _metrics

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_classification_iloc(ds, train_idx, valid_idx):
    train_X = ds.X.take(train_idx)
    train_y = ds.y.take(train_idx).to_numpy().reshape(-1)
    valid_X = ds.X.take(valid_idx)
    valid_w = ds.w.take(valid_idx).to_numpy().reshape(-1)
    valid_y = ds.y.take(valid_idx).to_numpy().reshape(-1)
    return valid_y

def gen_classification_pred_concat(accuracy_score, base_dfs, ds, f1_score, precision_score, pred, recall_score, roc_auc_score, valid_idx):

    pred_df = pl.DataFrame(
        {"index": ds.y.index[valid_idx], "pred": pred}
    )
    base_dfs.append(
        pred_df.join(
            base_df,
            on="index",
            how="inner",
        )
    )
    base_df = pl.concat(base_dfs)
    _metrics = {
        "roc_auc": roc_auc_score(base_df["y"], base_df["pred"]),
        "accuracy": accuracy_score(base_df["y"], base_df["pred"] > 0.5),
        "precision": precision_score(base_df["y"], base_df["pred"] > 0.5),
        "recall": recall_score(base_df["y"], base_df["pred"] > 0.5),
        "f1": f1_score(base_df["y"], base_df["pred"] > 0.5),
    }
    return _metrics

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: classification_pred_concat ===

# L1 smoke – generated
try:
    _r = gen_classification_pred_concat(FIX_CLASSIFICATION_PRED_CONCAT_ACCURACY_SCORE, make_classification_pred_concat_base_dfs_pl(), FIX_CLASSIFICATION_PRED_CONCAT_DS, FIX_CLASSIFICATION_PRED_CONCAT_F1_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_PRECISION_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_PRED, FIX_CLASSIFICATION_PRED_CONCAT_RECALL_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_ROC_AUC_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_VALID_IDX)
    print("✅ L1 smoke gen_classification_pred_concat: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_classification_pred_concat: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_classification_pred_concat(FIX_CLASSIFICATION_PRED_CONCAT_ACCURACY_SCORE, make_classification_pred_concat_base_dfs_pd(), FIX_CLASSIFICATION_PRED_CONCAT_DS, FIX_CLASSIFICATION_PRED_CONCAT_F1_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_PRECISION_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_PRED, FIX_CLASSIFICATION_PRED_CONCAT_RECALL_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_ROC_AUC_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_VALID_IDX)
    print("✅ L1 smoke before_classification_pred_concat: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_classification_pred_concat: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_classification_pred_concat(FIX_CLASSIFICATION_PRED_CONCAT_ACCURACY_SCORE, make_classification_pred_concat_base_dfs_pd(), FIX_CLASSIFICATION_PRED_CONCAT_DS, FIX_CLASSIFICATION_PRED_CONCAT_F1_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_PRECISION_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_PRED, FIX_CLASSIFICATION_PRED_CONCAT_RECALL_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_ROC_AUC_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_VALID_IDX)
    _rg = gen_classification_pred_concat(FIX_CLASSIFICATION_PRED_CONCAT_ACCURACY_SCORE, make_classification_pred_concat_base_dfs_pl(), FIX_CLASSIFICATION_PRED_CONCAT_DS, FIX_CLASSIFICATION_PRED_CONCAT_F1_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_PRECISION_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_PRED, FIX_CLASSIFICATION_PRED_CONCAT_RECALL_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_ROC_AUC_SCORE, FIX_CLASSIFICATION_PRED_CONCAT_VALID_IDX)
    compare(_rb, _rg, "classification_pred_concat")
except Exception as _e:
    print(f"❌ L2 equivalence classification_pred_concat: setup error — {type(_e).__name__}: {_e}")

# L3 edge - empty valid_idx on both sides.
try:
    _rb = before_classification_pred_concat(
        FIX_CLASSIFICATION_PRED_CONCAT_ACCURACY_SCORE,
        make_classification_pred_concat_base_dfs_pd(),
        FIX_CLASSIFICATION_PRED_CONCAT_DS,
        FIX_CLASSIFICATION_PRED_CONCAT_F1_SCORE,
        FIX_CLASSIFICATION_PRED_CONCAT_PRECISION_SCORE,
        np.array([]),
        FIX_CLASSIFICATION_PRED_CONCAT_RECALL_SCORE,
        FIX_CLASSIFICATION_PRED_CONCAT_ROC_AUC_SCORE,
        [],
    )
    _rg = gen_classification_pred_concat(
        FIX_CLASSIFICATION_PRED_CONCAT_ACCURACY_SCORE,
        make_classification_pred_concat_base_dfs_pl(),
        FIX_CLASSIFICATION_PRED_CONCAT_DS,
        FIX_CLASSIFICATION_PRED_CONCAT_F1_SCORE,
        FIX_CLASSIFICATION_PRED_CONCAT_PRECISION_SCORE,
        np.array([]),
        FIX_CLASSIFICATION_PRED_CONCAT_RECALL_SCORE,
        FIX_CLASSIFICATION_PRED_CONCAT_ROC_AUC_SCORE,
        [],
    )
    if _rb == _rg:
        print("✅ L3 edge classification_pred_concat empty idx: MATCH")
    else:
        print(f"❌ L3 edge classification_pred_concat empty idx: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge classification_pred_concat empty idx: {type(_e).__name__}: {_e}")

# AUDIT-41: verify the actual arrays supplied to every metric, not only the
# constant metric return values.
try:
    def _metric_set(call_log):
        def _metric(name):
            def _record(left, right, *args, **kwargs):
                call_log[name] = (
                    np.asarray(left).reshape(-1).tolist(),
                    np.asarray(right).reshape(-1).tolist(),
                )
                return 0.75
            return _record
        return {name: _metric(name) for name in ["accuracy", "f1", "precision", "recall", "roc_auc"]}

    _before_calls, _gen_calls = {}, {}
    _bm, _gm = _metric_set(_before_calls), _metric_set(_gen_calls)
    before_classification_pred_concat(
        _bm["accuracy"], make_classification_pred_concat_base_dfs_pd(),
        FIX_CLASSIFICATION_PRED_CONCAT_DS, _bm["f1"], _bm["precision"],
        FIX_CLASSIFICATION_PRED_CONCAT_PRED, _bm["recall"], _bm["roc_auc"],
        FIX_CLASSIFICATION_PRED_CONCAT_VALID_IDX,
    )
    gen_classification_pred_concat(
        _gm["accuracy"], make_classification_pred_concat_base_dfs_pl(),
        FIX_CLASSIFICATION_PRED_CONCAT_DS, _gm["f1"], _gm["precision"],
        FIX_CLASSIFICATION_PRED_CONCAT_PRED, _gm["recall"], _gm["roc_auc"],
        FIX_CLASSIFICATION_PRED_CONCAT_VALID_IDX,
    )
    if _before_calls == _gen_calls:
        print("✅ L2 equivalence classification_pred_concat metric inputs: MATCH")
    else:
        print(f"❌ L2 equivalence classification_pred_concat metric inputs: MISMATCH — before={_before_calls}, gen={_gen_calls}")
except Exception as _e:
    print(f"❌ L2 equivalence classification_pred_concat metric inputs: {type(_e).__name__}: {_e}")
